# Import

In [ ]:
import os
import bm25s
import numpy as np

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from beir.retrieval.models import SentenceBERT

# Add Data

In [ ]:
def add_data(dataset_name: str):
    url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset_name)
    out_dir = os.path.join(os.getcwd(), "datasets")
    data_path = util.download_and_unzip(url, out_dir)
    return data_path

# Add Models

## Add Model Dense

In [ ]:
def add_model_dense(model_name: str):
    dense_model = DRES(SentenceBERT(f"sentence-transformers/{model_name}"), batch_size=64)
    return dense_model

## Add model Sparse

In [ ]:
def add_model_sparse():
    retriever = bm25s.BM25(method="lucene", k1=1.2, b=0.75)
    return retriever

# Metrics

In [ ]:
def normalize_scores(results):
    """Min-Max normalization of scores for BM25 and Dense."""
    normalized = {}
    for qid, docs in results.items():
        if not docs:
            continue
        scores = list(docs.values())
        min_s, max_s = min(scores), max(scores)

        normalized[qid] = {}
        for did, score in docs.items():
            if max_s > min_s:
                normalized[qid][did] = (score - min_s) / (max_s - min_s)
            else:
                normalized[qid][did] = 0.5
    return normalized

def get_hybrid_scores(dense_res, sparse_res, alpha):
    """alpha * Dense + (1 - alpha) * Sparse."""
    hybrid = {}
    all_qids = set(dense_res.keys()) | set(sparse_res.keys())

    for qid in all_qids:
        hybrid[qid] = {}
        q_dense = dense_res.get(qid, {})
        q_sparse = sparse_res.get(qid, {})
        all_dids = set(q_dense.keys()) | set(q_sparse.keys())

        for did in all_dids:
            s_d = q_dense.get(did, 0.0)
            s_s = q_sparse.get(did, 0.0)
            hybrid[qid][did] = float(alpha * s_d + (1.0 - alpha) * s_s)

    return hybrid

In [ ]:
datasets = ["scifact", "fiqa", "nfcorpus"] # msmarco и scidocs
alphas = np.round(np.arange(0.0, 1.05, 0.05), 2)

results_ndcg = {ds: [] for ds in datasets}
results_mrr = {ds: [] for ds in datasets}

evaluator = EvaluateRetrieval()

for ds in datasets:
    print(f"\n[{ds.upper()}] Loading data...")
    data_path = add_data(ds)
    corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

    corpus_ids = list(corpus.keys())
    corpus_texts = [f"{corpus[cid].get('title', '')} {corpus[cid].get('text', '')}" for cid in corpus_ids]
    query_ids = [qid for qid in queries.keys() if qid in qrels]
    query_texts = [queries[qid] for qid in query_ids]

    print(f"[{ds.upper()}] Running Dense Retrieval...")
    dense_model = add_model_dense("all-MiniLM-L6-v2")
    dense_retriever = EvaluateRetrieval(dense_model, score_function="cos_sim")
    dense_raw = dense_retriever.retrieve(corpus, queries)

    print(f"[{ds.upper()}] Running Sparse Retrieval (BM25s)...")
    sparse_model = add_model_sparse()
    corpus_tokens = bm25s.tokenize(corpus_texts)
    sparse_model.index(corpus_tokens)

    query_tokens = bm25s.tokenize(query_texts)
    docs, scores = sparse_model.retrieve(query_tokens, corpus=corpus_ids, k=100)

    sparse_raw = {}
    for i, qid in enumerate(query_ids):
        sparse_raw[qid] = {docs[i][j]: float(scores[i][j]) for j in range(len(docs[i]))}

    print(f"[{ds.upper()}] Normalizing scores...")
    dense_norm = normalize_scores(dense_raw)
    sparse_norm = normalize_scores(sparse_raw)

    print(f"[{ds.upper()}] Evaluating alphas...")
    for alpha in alphas:
        hybrid_res = get_hybrid_scores(dense_norm, sparse_norm, alpha)

        ndcg_dict, _, _, _ = evaluator.evaluate(qrels, hybrid_res, [10])
        mrr_dict = evaluator.evaluate_custom(qrels, hybrid_res, [10], metric="mrr")
        results_ndcg[ds].append(ndcg_dict["NDCG@10"])

        mrr_key = list(mrr_dict.keys())[0]
        results_mrr[ds].append(mrr_dict[mrr_key])

print("\nBenchmark completed!")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for i, ds in enumerate(datasets):
    ax1.plot(alphas, results_ndcg[ds], label=ds.upper(), color=colors[i], marker='o', markersize=4)
    max_idx = np.argmax(results_ndcg[ds])
    ax1.scatter(alphas[max_idx], results_ndcg[ds][max_idx], color=colors[i], s=100, marker='*')

ax1.set_title("NDCG@10 by Dense Weight ($\\alpha$)", fontsize=14)
ax1.set_xlabel("$\\alpha$ (0 = Pure BM25, 1 = Pure Dense)", fontsize=12)
ax1.set_ylabel("NDCG@10", fontsize=12)
ax1.set_xticks(alphas)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

for i, ds in enumerate(datasets):
    ax2.plot(alphas, results_mrr[ds], label=ds.upper(), color=colors[i], marker='o', markersize=4)
    max_idx = np.argmax(results_mrr[ds])
    ax2.scatter(alphas[max_idx], results_mrr[ds][max_idx], color=colors[i], s=100, marker='*')

ax2.set_title("MRR@10 by Dense Weight ($\\alpha$)", fontsize=14)
ax2.set_xlabel("$\\alpha$ (0 = Pure BM25, 1 = Pure Dense)", fontsize=12)
ax2.set_ylabel("MRR@10", fontsize=12)
ax2.set_xticks(alphas)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

plt.tight_layout()
# plt.savefig("hybrid_benchmark.pdf", dpi=300, format='pdf')
plt.show()

# Train Model

In [ ]:
import torch
import torch.nn as nn

class AlphaRouter(nn.Module):
    def __init__(self, input_dim=4):
        super(AlphaRouter, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x_q):
        return self.net(x_q).squeeze(-1)